In [34]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [35]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google.colab import files
import re
import io

In [36]:
uploaded = files.upload()  # This opens a dialog to upload files

for name, file_obj in uploaded.items():
    markdown_text = io.StringIO(file_obj.decode('utf-8')).read()
    filename = name  # Keep for reference
    break  # just take the first file uploaded

with open(filename, "r", encoding="utf-8") as file:
    markdown_text = file.read()

Saving Task_Texts.txt to Task_Texts (3).txt


In [37]:
import re

def parse_markdown(markdown_text):
    lines = markdown_text.split('\n')
    elements = []

    for line in lines:
        stripped = line.strip()

        if stripped.startswith("# "):  # H1
            elements.append(('heading1', stripped[2:].strip()))

        elif stripped.startswith("## "):  # H2
            elements.append(('heading2', stripped[3:].strip()))

        elif stripped.startswith("### "):  # H3
            elements.append(('heading3', stripped[4:].strip()))

        elif re.match(r"^- \[ \]", stripped):  # Checkbox
            task_text = stripped[6:].strip()
            elements.append(('checkbox', task_text))

        elif re.match(r"^- ", stripped):  # Bullet point
            elements.append(('bullet', stripped[2:].strip()))

        elif stripped.startswith("*"):  # Possibly nested bullet point
            indent = len(re.match(r"^\*+", stripped).group(0)) - 1
            bullet_text = stripped.lstrip("*").strip()
            elements.append(('nested_bullet', indent, bullet_text))

        elif re.match(r"^---$", stripped):  # Horizontal rule or footer separator
            elements.append(('divider', ''))

        elif stripped != "":
            elements.append(('text', stripped))

    return elements

In [38]:
parsed_elements = parse_markdown(markdown_text)

for elem in parsed_elements:  # preview the first few
    print(elem)

('heading1', 'Product Team Sync - May 15, 2023')
('heading2', 'Attendees')
('bullet', 'Sarah Chen (Product Lead)')
('bullet', 'Mike Johnson (Engineering)')
('bullet', 'Anna Smith (Design)')
('bullet', 'David Park (QA)')
('heading2', 'Agenda')
('heading3', '1. Sprint Review')
('nested_bullet', 0, 'Completed Features')
('nested_bullet', 0, 'User authentication flow')
('nested_bullet', 0, 'Dashboard redesign')
('nested_bullet', 0, 'Performance optimization')
('nested_bullet', 0, 'Reduced load time by 40%')
('nested_bullet', 0, 'Implemented caching solution')
('nested_bullet', 0, 'Pending Items')
('nested_bullet', 0, 'Mobile responsive fixes')
('nested_bullet', 0, 'Beta testing feedback integration')
('heading3', '2. Current Challenges')
('nested_bullet', 0, 'Resource constraints in QA team')
('nested_bullet', 0, 'Third-party API integration delays')
('nested_bullet', 0, 'User feedback on new UI')
('nested_bullet', 0, 'Navigation confusion')
('nested_bullet', 0, 'Color contrast issues')
('

In [39]:
auth.authenticate_user()
docs_service = build('docs', 'v1')
drive_service = build('drive', 'v3')

doc = docs_service.documents().create(body={"title": "Product Team Sync - Markdown Converted"}).execute()
document_id = doc.get('documentId')
print("Created document with ID:", document_id)

Created document with ID: 1D4BWEOcfhho8GdjyTtAT1wrrJCG2GlgAt7B2E_cWVeM


In [40]:
def build_google_docs_requests(parsed_elements):
    requests = []
    current_index = 1  # Google Docs index starts from 1

    for item in parsed_elements:
        if item[0] in ('heading1', 'heading2', 'heading3'):
            style_map = {
                'heading1': 'HEADING_1',
                'heading2': 'HEADING_2',
                'heading3': 'HEADING_3'
            }
            text = item[1] + '\n'
            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": text
                }
            })
            requests.append({
                "updateParagraphStyle": {
                    "range": {
                        "startIndex": current_index,
                        "endIndex": current_index + len(text)
                    },
                    "paragraphStyle": {
                        "namedStyleType": style_map[item[0]]
                    },
                    "fields": "namedStyleType"
                }
            })
            current_index += len(text)

        elif item[0] == 'bullet':
            text = item[1] + '\n'
            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": text
                }
            })
            requests.append({
                "createParagraphBullets": {
                    "range": {
                        "startIndex": current_index,
                        "endIndex": current_index + len(text)
                    },
                    "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE"
                }
            })
            current_index += len(text)

        elif item[0] == 'nested_bullet':
            indent_level, text = item[1], item[2] + '\n'
            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": text
                }
            })
            requests.append({
                "createParagraphBullets": {
                    "range": {
                        "startIndex": current_index,
                        "endIndex": current_index + len(text)
                    },
                    "bulletPreset": "BULLET_DISC_CIRCLE_SQUARE"
                }
            })
            requests.append({
                "updateParagraphStyle": {
                    "range": {
                        "startIndex": current_index,
                        "endIndex": current_index + len(text)
                    },
                    "paragraphStyle": {
                        "indentStart": {"magnitude": 18 * indent_level, "unit": "PT"}
                    },
                    "fields": "indentStart"
                }
            })
            current_index += len(text)

        elif item[0] == 'checkbox':
            # Prepend the Unicode empty checkbox (☐) followed by a space.
            text = "☐ " + item[1] + '\n'
            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": text
                }
            })
            current_index += len(text)

        elif item[0] == 'text':
            text = item[1] + '\n'
            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": text
                }
            })
            current_index += len(text)

        elif item[0] == 'divider':
            divider_text = "────────────────────────────\n"
            requests.append({
                "insertText": {
                    "location": {"index": current_index},
                    "text": divider_text
                }
            })
            current_index += len(divider_text)

    return requests


In [41]:
requests = build_google_docs_requests(parsed_elements)

docs_service.documents().batchUpdate(
    documentId=document_id,
    body={'requests': requests}
).execute()


{'replies': [{},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {},
  {}],
 'writeControl': {'requiredRevisionId': 'ALBJ4Lsm5SGskibccDn-vT8e27CPDAtbKwkVQVNhyQjxvaJFTovKpTs6Ruw7IAPFaCiclHk8Ozf8V9SNL1Ln9w'},
 'documentId': '1D4BWEOcfhho8GdjyTtAT1wrrJCG2GlgAt7B2E_cWVeM'}

In [42]:
doc_url = f"https://docs.google.com/document/d/{document_id}/edit"

print(f"Your document is ready here at {doc_url}")

Your document is ready here at https://docs.google.com/document/d/1D4BWEOcfhho8GdjyTtAT1wrrJCG2GlgAt7B2E_cWVeM/edit
